#  TP 4 : Étude empirique : les pratiques de refactoring avec les agents IA
## MGL804 – Réalisation et maintenance des logiciels

**Objectifs :**
1. **Partie 1** — Quantifier la prévalence des suggestions de refactoring dans les commentaires de révision humains sur des PRs agentiques.
2. **Partie 2** — Vérifier avec RefactoringMiner si les agents appliquent ces suggestions.

**Jeu de données :** [AIDev](https://huggingface.co/datasets/hao-li/AIDev) (sous-ensemble AIDev-pop)

> ⚠️ Ce notebook est un **guide de laboratoire**. Il fournit la structure, des indices et des points de départ — **c'est à vous d'écrire le code**. Les cellules marquées `# TODO` ou `# VOTRE CODE ICI` sont à compléter.\n",

> 🚨 **Avertissement important :** Les indices fournis dans ce notebook (noms de colonnes, noms de variables, exemples de jointures) sont indicatifs et peuvent ne pas correspondre exactement à la structure réelle des données. **Vous devez explorer les tables vous-même, vérifier les noms de colonnes réels avec `.columns` ou `.head()`, et adapter votre code en conséquence — ne copiez pas les indices sans les valider au préalable.**

---
## 0. Installation et configuration

Exécutez cette cellule une seule fois pour installer les dépendances.

In [ ]:
# Décommentez si nécessaire :
# !pip install pandas pyarrow matplotlib seaborn scipy tqdm datasets nbformat python-dotenv

### 0.1 Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import json
import subprocess
import os
from dotenv import load_dotenv
from tqdm.auto import tqdm
from scipy import stats

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 120)
sns.set_theme(style="whitegrid")

print("Imports réussis")

### 0.2 Configuration de Java et RefactoringMiner

RefactoringMiner 3.x requiert **Java 17+**. Si votre serveur a une version plus ancienne, installez-la via conda.

> **Indice :** Vérifiez votre version Java avec `!java -version`. Si c'est Java 8 ou 11, vous devez installer Java 17.

In [ ]:
# Vérifiez votre version Java actuelle
!java -version

In [ ]:
# Si Java < 17, décommentez et exécutez :
# !conda install -c conda-forge openjdk=17 -y

# Puis configurez le PATH :
# import os
# conda_base = os.popen('conda info --base').read().strip()
# java17_home = f"{conda_base}/lib/jvm"
# os.environ['JAVA_HOME'] = java17_home
# os.environ['PATH'] = f"{java17_home}/bin:{os.environ['PATH']}"
# !java -version

In [ ]:
# Token GitHub pour l'API (nécessaire pour le mode -gc / -gp de RefactoringMiner)
# Le token est chargé depuis le fichier .env (jamais commité dans git)
# Créez un .env à la racine du projet avec : GITHUB_OAUTH=ghp_...

load_dotenv()

if not os.environ.get('GITHUB_OAUTH'):
    raise ValueError("GITHUB_OAUTH manquant — ajoutez-le dans votre fichier .env")

# ─── Configuration RefactoringMiner ───────────────────────────────────────────
import subprocess
RMINER_LIB = r"c:\Repos\RefactoringMiner-3.1.3\lib"
RMINER_CMD = ["java", "-cp", RMINER_LIB + "\\*", "org.refactoringminer.RefactoringMiner"]

result = subprocess.run(RMINER_CMD + ["-h"], capture_output=True, text=True, timeout=30)
if "Show options" in result.stdout or "RefactoringMiner" in result.stdout:
    print("✅ RefactoringMiner 3.1.3 opérationnel")
else:
    print("❌ Problème RefactoringMiner — stdout:", result.stdout[:200])
    print("   stderr:", result.stderr[:200])

---
## 1. Chargement des données AIDev

Nous chargeons les tables Parquet directement depuis Hugging Face.

> **Indice :** Après le premier chargement, sauvegardez localement avec `df.to_parquet("local_file.parquet")` pour accélérer les exécutions suivantes.

In [ ]:
print("Chargement des pull requests...")
df_prs = pd.read_parquet("hf://datasets/hao-li/AIDev/pull_request.parquet")

print("Chargement des commentaires de révision (v2)...")
df_review_comments = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_review_comments_v2.parquet")

print("Chargement des commits par PR...")
df_commits = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_commits.parquet")

print("Chargement des détails de commits...")
df_commit_details = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_commit_details.parquet")

print("Chargement de la timeline...")
df_timeline = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_timeline.parquet")

print("Chargement des reviews...")
df_reviews = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_reviews.parquet")

print("Chargement des dépôts...")
df_repos = pd.read_parquet("hf://datasets/hao-li/AIDev/repository.parquet")

print("\n✅ Toutes les tables sont chargées.")

### 1.1 Exploration rapide des données

Prenez le temps d'explorer la structure de chaque table avant de commencer l'analyse.

> **Indice :** Pour chaque table, examinez les colonnes, le nombre de lignes, et quelques lignes avec `.head()`. Identifiez les clés de jointure potentielles (`id`, `pr_id`, `repo_id`, etc.).

In [ ]:
# Aperçu des tables et de leurs colonnes
tables = {
    "pull_request": df_prs,
    "pr_review_comments_v2": df_review_comments,
    "pr_commits": df_commits,
    "pr_commit_details": df_commit_details,
    "pr_timeline": df_timeline,
    "pr_reviews": df_reviews,
    "repository": df_repos,
}

for name, df in tables.items():
    print(f"\n{'='*60}")
    print(f"{name} — {len(df):,} lignes, {len(df.columns)} colonnes")
    print(f"   Colonnes : {list(df.columns)}")

In [ ]:
# TODO : Explorez les tables individuellement
# Essayez : df_prs.head(3), df_review_comments.head(3), df_repos.head(3)
# Repérez les colonnes clés pour les jointures à venir.

# VOTRE CODE ICI

---
## Partie 1 : Prévalence des suggestions de refactoring (13 pts)

### Étape 1.1 — Filtrer les PRs agentiques

Les 5 agents connus dans AIDev sont : **Codex, Devin, Copilot, Cursor, Claude Code**.

> **Indice :** Cherchez une colonne dans `df_prs` qui identifie l'agent (ex. `'agent'`). Affichez ses valeurs uniques avec `.unique()` pour connaître les noms exacts, puis filtrez avec `.isin()`.

In [ ]:
# TODO : Identifiez la colonne qui indique l'agent et affichez ses valeurs uniques

# VOTRE CODE ICI

In [ ]:
# TODO : Filtrez les PRs agentiques en utilisant les noms exacts trouvés ci-dessus
#
# INDICE :
#   AGENTS = ['OpenAI_Codex', 'Copilot', 'Devin', 'Cursor', 'Claude_Code']  # vérifiez les noms exacts
#   df_agentic = df_prs[df_prs['agent'].isin(AGENTS)]
#
# Affichez le nombre de PRs par agent avec .value_counts()

# VOTRE CODE ICI

### Étape 1.2 — Joindre avec les commentaires de révision

Fusionnez les PRs agentiques avec les commentaires inline. **Gardez tous les commentaires** (humains et bots), puis analysez la distribution des auteurs (humain vs IA/bot).

> **Indice :** Il n'y a pas de clé directe entre `df_review_comments` et `df_prs`. Vous devez passer par `df_reviews` comme table intermédiaire :
> 1. `df_review_comments` → `df_reviews` (via `pull_request_review_id` ↔ `id`)
> 2. Résultat → `df_prs` (via `pr_id` ↔ `id`)
>
> Utilisez `.merge()` avec le paramètre `suffixes` pour éviter les conflits de noms de colonnes.
>
> Après la jointure, explorez la colonne auteur pour distinguer les commentaires humains des commentaires bot/IA et affichez la distribution (ex. `.value_counts()`).

In [ ]:
# TODO : Trouvez les colonnes communes entre les tables
print("Colonnes PR      :", sorted(df_prs.columns))
print("Colonnes Reviews :", sorted(df_reviews.columns))
print("Colonnes Comments:", sorted(df_review_comments.columns))

In [ ]:
# TODO : Effectuez la jointure en deux étapes
#
# INDICE :
#   Étape A : review_comments + reviews
#     df_merged = df_review_comments.merge(df_reviews,
#         left_on='pull_request_review_id', right_on='id',
#         how='left', suffixes=('_review_comment', '_review'))
#
#   Étape B : résultat + pull_request
#     df_comments = df_merged.merge(df_prs,
#         left_on='pr_id', right_on='id',
#         how='left', suffixes=('_comment', '_pr'))
#
# Vérifiez avec df_comments.shape et df_comments.head(3)

# VOTRE CODE ICI

In [ ]:
# TODO : Analysez la distribution des commentaires humains vs bots/IA
#
# INDICES :
#   1. Cherchez la colonne auteur : [c for c in df_comments.columns if 'user' in c.lower() or 'author' in c.lower()]
#   2. Identifiez les bots (souvent le nom contient '[bot]' ou 'github-actions')
#   3. Affichez la distribution : combien de commentaires humains vs bots ?
#   4. Visualisez avec un barplot ou un camembert
#
# NOTE : Gardez TOUS les commentaires dans df_comments pour la suite de l'analyse.

# VOTRE CODE ICI

### Étape 1.3 — Identifier les commentaires de refactoring (mots-clés)

Appliquez le dictionnaire de mots-clés fourni dans l'énoncé.

> **Indice :** Combinez les regex en un seul pattern avec `'|'.join(patterns)`, puis appliquez `.str.contains()` sur la colonne body des commentaires. Attention au paramètre `na=False` pour gérer les valeurs manquantes.

In [ ]:
import re

# Patterns de détection de refactoring (fournis dans l'énoncé)
KEYWORD_PATTERNS = {
    "Add*"                         : r"\badd\w*",
    "Chang*"                       : r"\bchang\w*",
    "Chang* the name"              : r"\bchang\w*\s+the\s+name",
    "Cleanup"                      : r"\bcleanup\b",
    "Clean* up"                    : r"\bclean\w*\s+up",
    "Code clarity"                 : r"\bcode\s+clarity",
    "Code clean*"                  : r"\bcode\s+clean\w*",
    "Code organization"            : r"\bcode\s+organization",
    "Code review"                  : r"\bcode\s+review",
    "Clean code"                   : r"\bclean\s+code",
    "Creat*"                       : r"\bcreat\w*",
    "Customiz*"                    : r"\bcustomiz\w*",
    "Easier to maintain"           : r"\beasier\s+to\s+maintain",
    "Encapsulat*"                  : r"\bencapsulat\w*",
    "Enhanc*"                      : r"\benhanc\w*",
    "Extend*"                      : r"\bextend\w*",
    "Extract*"                     : r"\bextract\w*",
    "Fix*"                         : r"\bfix\w*",
    "Inlin*"                       : r"\binlin\w*",
    "Improv*"                      : r"\bimprov\w*",
    "Improv* code quality"         : r"\bimprov\w*\s+code\s+quality",
    "Introduc*"                    : r"\bintroduc\w*",
    "Merg*"                        : r"\bmerg\w*",
    "Modif*"                       : r"\bmodif\w*",
    "Modulariz*"                   : r"\bmodulariz\w*",
    "Migrat*"                      : r"\bmigrat\w*",
    "Mov*"                         : r"\bmov\w*",
    "Organiz*"                     : r"\borganiz\w*",
    "Polish*"                      : r"\bpolish\w*",
    "Reduc*"                       : r"\breduc\w*",
    "Refactor*"                    : r"\brefactor\w*",
    "Refin*"                       : r"\brefin\w*",
    "Remov*"                       : r"\bremov\w*",
    "Remov* redundant code"        : r"\bremov\w*\s+redundant\s+code",
    "Renam*"                       : r"\brenam\w*",
    "Remov* unused dependencies"   : r"\bremov\w*\s+unused\s+dependenc\w*",
    "Reorganiz*"                   : r"\breorganiz\w*",
    "Replac*"                      : r"\breplac\w*",
    "Restructur*"                  : r"\brestructur\w*",
    "Rework*"                      : r"\brework\w*",
    "Rewrit*"                      : r"\brewrit\w*",
    "Simplif*"                     : r"\bsimplif\w*",
    "Split*"                       : r"\bsplit\w*",
}

combined_pattern = '|'.join(KEYWORD_PATTERNS.values())
print(f"Pattern combiné ({len(KEYWORD_PATTERNS)} mots-clés)")

In [ ]:
# TODO : Appliquez le pattern sur le body des commentaires
#
# INDICES :
#   1. Identifiez la bonne colonne body : [c for c in df_comments.columns if 'body' in c.lower()]
#   2. Créez une colonne booléenne 'is_refactoring' avec .str.contains()
#   3. Affichez le nombre et pourcentage de commentaires de refactoring

# 1. Identification de la bonne colonne body
body_cols = [c for c in df_review_comments.columns if 'body' in c.lower()]
print(f"Colonnes 'body' trouvées : {body_cols}")
body_col = body_cols[0]

# 2. Création de la colonne booléenne 'is_refactoring' avec .str.contains()

df_review_comments['is_refactoring'] = df_review_comments[body_col].str.contains(
    combined_pattern,
    flags=re.IGNORECASE,
    regex=True,
    na=False
)

# 3. Affichages des stats
n_total = len(df_review_comments)
n_refactoring = df_review_comments['is_refactoring'].sum()
pct = n_refactoring / n_total * 100

print(f"\nTotal commentaires       : {n_total:,}")
print(f"Commentaires refactoring : {n_refactoring:,}")
print(f"Pourcentage              : {pct:.2f}%")

### Étape 1.4 — Calculer la prévalence par agent

> **Indice :** Utilisez `.groupby('agent').agg()` pour compter le total et la somme des `is_refactoring`, puis calculez le pourcentage.

In [ ]:
# TODO : Tableau récapitulatif de prévalence par agent
#
# INDICE :
#   prevalence = df_comments.groupby('agent').agg(
#       total_comments=('is_refactoring', 'count'),
#       refactoring_comments=('is_refactoring', 'sum'),
#   )
#   prevalence['percentage'] = ...

# VOTRE CODE ICI

### Étape 1.5 — Taxonomie et distribution par catégorie

Classifiez chaque commentaire de refactoring dans l'une des 6 catégories.

> **Indice :** Créez un dictionnaire `CATEGORIES` qui associe chaque catégorie à une liste de regex. Écrivez une fonction `classify_refactoring(text)` qui retourne la première catégorie correspondante, ou `'Autre'` si aucune ne correspond. Appliquez-la avec `.apply()`.
>
> Catégories suggérées : Changements structurels, Nommage/lisibilité, Code mort/nettoyage, Organisation du code, Duplication, Remplacement/migration.

In [ ]:
# TODO : Définissez les catégories et leurs patterns
#
# INDICE :
#   CATEGORIES = {
#       'Changements structurels': [r'\bextract\b', r'\binline\b', r'\bsplit\b', ...],
#       'Nommage / lisibilité':    [r'\brename', r'\bclarity', ...],
#       ...
#   }
#
#   def classify_refactoring(text):
#       # Itérez sur CATEGORIES, retournez la première catégorie matchée
#       ...
#
#   df_refactoring = df_comments[df_comments['is_refactoring']].copy()
#   df_refactoring['category'] = df_refactoring[body_col].apply(classify_refactoring)

CATEGORIES = {
    'Changements structurels': [
        r'\bextract\w*', r'\binlin\w*', r'\bsplit\w*', r'\bmerg\w*',
        r'\brestructur\w*', r'\brework\w*', r'\brewrit\w*', r'\brefactor\w*',
        r'\bmov\w*', r'\bmigrat\w*', r'\bmodulariz\w*',
    ],
    'Nommage / lisibilité': [
        r'\brenam\w*', r'\bcode\s+clarity', r'\bclean\s+code',
        r'\bcode\s+clean\w*', r'\beasier\s+to\s+maintain',
        r'\bsimplif\w*', r'\brefin\w*', r'\bpolish\w*', r'\borganiz\w*',
    ],
    'Code mort / nettoyage': [
        r'\bremov\w*\s+unused\s+dependenc\w*', r'\bremov\w*\s+redundant\s+code',
        r'\bcleanup\b', r'\bclean\w*\s+up', r'\bremov\w*', r'\breduc\w*',
    ],
    'Organisation du code': [
        r'\bcode\s+organization', r'\breorganiz\w*', r'\bcode\s+review',
        r'\bmodif\w*', r'\borganiz\w*',
    ],
    'Duplication': [
        r'\bduplic\w*', r'\brepeat\w*', r'\bredundant\w*', r'\breuse\w*',
        r'\bcommon\w*', r'\bshare\w*', r'\bconsolid\w*',
    ],
    'Remplacement / migration': [
        r'\breplac\w*', r'\bmigrat\w*', r'\bupgrad\w*', r'\bswitch\w*',
        r'\bconvert\w*', r'\btransform\w*', r'\bport\w*',
    ],
}

#categories etablies avec KEYWORD_PATTERNS

def classify_refactoring(text):
    if not isinstance(text, str):
        return 'Autre'
    for category, patterns in CATEGORIES.items():
        combined = '|'.join(patterns)
        if re.search(combined, text, flags=re.IGNORECASE):
            return category
    return 'Autre'

df_refactoring = df_review_comments[df_review_comments['is_refactoring']].copy()
df_refactoring['category'] = df_refactoring[body_col].apply(classify_refactoring)

print(f"Commentaires classifiés : {len(df_refactoring):,}")
print(df_refactoring['category'].value_counts())

In [ ]:
# TODO : Diagramme en barres — distribution par catégorie
#
# INDICE : df_refactoring['category'].value_counts().plot(kind='barh')

import matplotlib.pyplot as plt

df_refactoring['category'].value_counts().plot(kind='barh', figsize=(10, 5), color='steelblue')

plt.xlabel("Nombre de commentaires")
plt.title("Distribution des commentaires de refactoring par catégorie")
plt.tight_layout()
plt.show()

### Étape 1.6 — Analyse par agent et par langage

> **Indice :** Joignez `df_refactoring` avec `df_repos` pour récupérer la colonne `language`. Utilisez `pd.crosstab()` pour un tableau croisé agent × langage. Visualisez avec un barplot empilé.

In [ ]:
# TODO : Analyse par agent et par langage
#
# INDICES :
#   1. Joignez df_refactoring avec df_repos sur repo_id
#   2. pd.crosstab(df_refactoring_lang['agent'], df_refactoring_lang['language'])
#   3. Visualisez les 8 langages les plus fréquents avec .plot(kind='bar', stacked=True)

# VOTRE CODE ICI

### Étape 1.7 — Validation manuelle (précision / rappel)

Les commentaires à annoter vous seront envoyés sur Discord.

> **Indice :**
> 1. Exportez en CSV votre annotation manuelle
> 2. Après annotation, calculez : `precision = TP / (TP + FP)` et `recall = TP / (TP + FN)`

In [ ]:
# TODO : Échantillonnage et validation manuelle
#
# INDICES :
#   sample_positive = df_comments[df_comments['is_refactoring']].sample(50, random_state=42)
#   sample_negative = df_comments[~df_comments['is_refactoring']].sample(50, random_state=42)
#   sample_positive[[body_col]].to_csv('validation_positive.csv', index=False)
#
#   Après annotation :
#   TP = ...  # vrais positifs
#   FP = ...  # faux positifs
#   FN = ...  # faux négatifs
#   precision = TP / (TP + FP)
#   recall    = TP / (TP + FN)

import pandas as pd

# Chargement du fichier d'annotation
df_annotation = pd.read_parquet(
    'outputs/df_comments_agentic.parquet',
)
print(df.head())

###print("TEST####", [c for c in df_annotation.columns if 'keyword' in c.lower() or 'match' in c.lower()])

# Nettoyage : garder uniquement les lignes annotées (Consensus Final renseigné)
df_annotation = df_annotation[df_annotation['Consensus Final'].isin(['OUI', 'NON'])].copy()

# Alignement des colonnes :
# - has_keyword_match  → prédiction du modèle  (OUI = prédit refactoring)
# - Consensus Final    → vérité terrain         (OUI = vrai refactoring)

df_annotation['predicted'] = df_annotation['comments_with_keywords'] > 0
df_annotation['actual']    = df_annotation['Consensus Final'] == 'OUI'

# Calcul TP, FP, FN
TP = ((df_annotation['predicted'] == True)  & (df_annotation['actual'] == True)).sum()
FP = ((df_annotation['predicted'] == True)  & (df_annotation['actual'] == False)).sum()
FN = ((df_annotation['predicted'] == False) & (df_annotation['actual'] == True)).sum()
TN = ((df_annotation['predicted'] == False) & (df_annotation['actual'] == False)).sum()

# Métriques
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Échantillon annoté : {len(df_annotation)} commentaires")
print(f"\nMatrice de confusion :")
print(f"  TP (vrais positifs)  : {TP}")
print(f"  FP (faux positifs)   : {FP}")
print(f"  FN (faux négatifs)   : {FN}")
print(f"  TN (vrais négatifs)  : {TN}")
print(f"\nPrécision  : {precision:.2%}")
print(f"Rappel     : {recall:.2%}")
print(f"F1-score   : {f1:.2%}")

---
## Partie 2 : Application des suggestions avec RefactoringMiner (12 pts + 2 bonus)

> **Prérequis :** RefactoringMiner doit être installé et configuré (voir l'énoncé).
> Cette partie se concentre sur les projets **Java** et **Python** (supportés par RefactoringMiner 3.0).

### Étape 2.1 — Filtrer les projets Java et Python

> **Indice :** Filtrez `df_repos` par langage, puis joignez avec `df_agentic` pour obtenir les PRs agentiques dans ces langages.

In [ ]:
# ─── Chargement des outputs (étapes 1.1, 1.2, 2.1 déjà faites) ───
import pandas as pd
import os

OUTPUTS_DIR = "outputs"  # adapte si ton dossier est ailleurs

df_agentic = pd.read_parquet(os.path.join(OUTPUTS_DIR, "df_agentic.parquet"))
df_comments_agentic = pd.read_parquet(os.path.join(OUTPUTS_DIR, "df_comments_agentic.parquet"))
df_prs_java_python = pd.read_parquet(os.path.join(OUTPUTS_DIR, "df_prs_java_python.parquet"))

print(f"✅ df_agentic         : {len(df_agentic):,} PRs agentiques")
print(f"✅ df_comments_agentic: {len(df_comments_agentic):,} commentaires")
print(f"✅ df_prs_java_python : {len(df_prs_java_python):,} PRs Java/Python")

In [ ]:
# Étape 2.1 — déjà réalisée (tp4_guide_steps.py)
# On charge directement le résultat depuis le parquet

# Tableau croisé language × agent (question 2.1.a)
crosstab_21 = pd.crosstab(
    df_prs_java_python['language'],
    df_prs_java_python['agent']
)
print("PRs agentiques Java/Python par agent :")
print(crosstab_21)
print(f"\nTotal : {len(df_prs_java_python):,} PRs")

### Étape 2.2 — Sélectionner les PRs avec commentaires de refactoring

> **Indice :** Récupérez les `pr_id` uniques de `df_refactoring`, puis filtrez `df_prs_jp` pour ne garder que les PRs qui ont au moins un commentaire de refactoring.

In [ ]:
import re

# Patterns de détection de refactoring (identiques à la cellule 1.3)
KEYWORD_PATTERNS_22 = {
    "Add*"                         : r"\badd\w*",
    "Chang*"                       : r"\bchang\w*",
    "Chang* the name"              : r"\bchang\w*\s+the\s+name",
    "Cleanup"                      : r"\bcleanup\b",
    "Clean* up"                    : r"\bclean\w*\s+up",
    "Code clarity"                 : r"\bcode\s+clarity",
    "Code clean*"                  : r"\bcode\s+clean\w*",
    "Code organization"            : r"\bcode\s+organization",
    "Code review"                  : r"\bcode\s+review",
    "Clean code"                   : r"\bclean\s+code",
    "Creat*"                       : r"\bcreat\w*",
    "Customiz*"                    : r"\bcustomiz\w*",
    "Easier to maintain"           : r"\beasier\s+to\s+maintain",
    "Encapsulat*"                  : r"\bencapsulat\w*",
    "Enhanc*"                      : r"\benhanc\w*",
    "Extend*"                      : r"\bextend\w*",
    "Extract*"                     : r"\bextract\w*",
    "Fix*"                         : r"\bfix\w*",
    "Inlin*"                       : r"\binlin\w*",
    "Improv*"                      : r"\bimprov\w*",
    "Improv* code quality"         : r"\bimprov\w*\s+code\s+quality",
    "Introduc*"                    : r"\bintroduc\w*",
    "Merg*"                        : r"\bmerg\w*",
    "Modif*"                       : r"\bmodif\w*",
    "Modulariz*"                   : r"\bmodulariz\w*",
    "Migrat*"                      : r"\bmigrat\w*",
    "Mov*"                         : r"\bmov\w*",
    "Organiz*"                     : r"\borganiz\w*",
    "Polish*"                      : r"\bpolish\w*",
    "Reduc*"                       : r"\breduc\w*",
    "Refactor*"                    : r"\brefactor\w*",
    "Refin*"                       : r"\brefin\w*",
    "Remov*"                       : r"\bremov\w*",
    "Remov* redundant code"        : r"\bremov\w*\s+redundant\s+code",
    "Renam*"                       : r"\brenam\w*",
    "Remov* unused dependencies"   : r"\bremov\w*\s+unused\s+dependenc\w*",
    "Reorganiz*"                   : r"\breorganiz\w*",
    "Replac*"                      : r"\breplac\w*",
    "Restructur*"                  : r"\brestructur\w*",
    "Rework*"                      : r"\brework\w*",
    "Rewrit*"                      : r"\brewrit\w*",
    "Simplif*"                     : r"\bsimplif\w*",
    "Split*"                       : r"\bsplit\w*",
}

def has_refactoring_keyword(text):
    if not isinstance(text, str):
        return False
    text_lower = text.lower()
    return any(re.search(pattern, text_lower) for pattern in KEYWORD_PATTERNS_22.values())

# Appliquer le filtre sur les commentaires
df_comments_agentic['is_refactoring'] = df_comments_agentic['body_comment'].apply(has_refactoring_keyword)

# PRs Java/Python avec au moins 1 commentaire de refactoring
pr_ids_with_refactoring = set(
    df_comments_agentic[df_comments_agentic['is_refactoring']]['pr_id'].unique()
)

df_prs_refactoring = df_prs_java_python[
    df_prs_java_python['id_pr'].isin(pr_ids_with_refactoring)
].copy()

print(f"PRs Java/Python avec ≥1 commentaire refactoring : {len(df_prs_refactoring):,}")
print()
print("Tableau croisé langage × agent :")
print(pd.crosstab(df_prs_refactoring['language'], df_prs_refactoring['agent']))

### Étape 2.3 — Construire la chronologie par PR

Pour chaque PR, identifiez les **commits de suivi** : commits effectués **après** un commentaire de refactoring.

```
Temps ──────────────────────────────────────────────>

  commit_1     comment_refactoring     commit_2 (suivi)     commit_3 (suivi)
     │                 │                    │                     │
     ▼                 ▼                    ▼                     ▼
  ─────────────────────────────────────────────────────────────────
```

> **Indices :**
> - La table `pr_timeline` contient les événements : `event='committed'` pour les commits, avec `commit_id` = SHA et `created_at` = timestamp
> - Certains événements `committed` n'ont pas de `created_at` → utilisez `.bfill()` pour propager le timestamp du prochain événement
> - Comparez les timestamps des commentaires de refactoring avec ceux des commits pour identifier les commits de suivi

In [ ]:
# Étape 2.3 — Charger et préparer la timeline

TIMELINE_CACHE = os.path.join(OUTPUTS_DIR, "df_timeline.parquet")

if os.path.exists(TIMELINE_CACHE):
    print("Chargement local de pr_timeline...")
    df_timeline = pd.read_parquet(TIMELINE_CACHE)
else:
    print("Téléchargement de pr_timeline depuis Hugging Face (~1 min)...")
    df_timeline = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_timeline.parquet")
    df_timeline.to_parquet(TIMELINE_CACHE, index=False)
    print(f"✅ Sauvegardé localement : {TIMELINE_CACHE}")

print(f"✅ pr_timeline : {len(df_timeline):,} événements")
print(f"   Colonnes : {list(df_timeline.columns)}")

# Convertir created_at sur TOUTE la timeline (avant de filtrer)
df_timeline['created_at'] = pd.to_datetime(df_timeline['created_at'], utc=True, errors='coerce')

# bfill sur TOUTE la timeline groupée par pr_id
# (les événements non-committed ont des timestamps qui remplissent les NaT des committed)
df_timeline['created_at'] = df_timeline.groupby('pr_id')['created_at'].transform(
    lambda x: x.ffill().bfill()
)

# Maintenant filtrer sur committed
df_tl_commits = df_timeline[df_timeline['event'] == 'committed'].copy()

non_nat = df_tl_commits['created_at'].notna().sum()
print(f"\nÉvénements committed : {len(df_tl_commits):,}")
print(f"Avec timestamp valide : {non_nat:,} ({100*non_nat/len(df_tl_commits):.1f}%)")
print(df_tl_commits[['pr_id', 'event', 'commit_id', 'created_at']].head(5))

In [ ]:
# Étape 2.3 — Calcul des commits de suivi (approche vectorisée)

# 1. Filtrer les commentaires de refactoring pour les 684 PRs seulement
pr_ids_684 = set(df_prs_refactoring['id_pr'].tolist())

df_ref = df_comments_agentic[
    df_comments_agentic['is_refactoring'] &
    df_comments_agentic['pr_id'].isin(pr_ids_684)
].copy()
df_ref['submitted_at'] = pd.to_datetime(df_ref['submitted_at'], utc=True, errors='coerce')
df_ref = df_ref.dropna(subset=['submitted_at'])
print(f"Commentaires de refactoring dans les 684 PRs : {len(df_ref):,}")

# Identifier les bonnes colonnes
id_col   = next((c for c in df_ref.columns if c in ['id_comment', 'id']), None)
path_col = next((c for c in df_ref.columns if 'path' in c.lower()), None)
print(f"  Colonne id commentaire : {id_col}")
print(f"  Colonne path           : {path_col}")

# 2. Commits de la timeline pour les 684 PRs seulement
df_tl_684 = df_tl_commits[df_tl_commits['pr_id'].isin(pr_ids_684)].copy()
print(f"Commits timeline pour les 684 PRs : {len(df_tl_684):,}")

# 3. Jointure comments × commits sur pr_id
df_cross = df_ref[['pr_id', id_col, path_col, 'submitted_at']].merge(
    df_tl_684[['pr_id', 'commit_id', 'created_at']],
    on='pr_id',
    how='left'
)

# 4. Garder seulement les commits APRÈS le commentaire
df_cross = df_cross[
    df_cross['created_at'].notna() &
    (df_cross['created_at'] > df_cross['submitted_at'])
]

# 5. Agréger par commentaire
df_followups = (
    df_cross
    .groupby(['pr_id', id_col, path_col])
    .agg(followup_shas=('commit_id', lambda x: list(x.dropna().unique())),
         n_followups=('commit_id', 'nunique'))
    .reset_index()
    .rename(columns={id_col: 'comment_id', path_col: 'comment_path'})
)

print(f"\n✅ Résultats :")
print(f"   Commentaires avec commits de suivi : {len(df_followups):,}")
if not df_followups.empty:
    print(f"   Moyenne commits de suivi : {df_followups['n_followups'].mean():.2f}")
    print(df_followups[['pr_id', 'comment_path', 'n_followups']].head(10))

### Étape 2.4 — Exécuter RefactoringMiner

> **Indices :**
> - Le chemin vers l'exécutable dépend de votre installation. Vérifiez avec `!ls /chemin/vers/RefactoringMiner`
> - Mode `-gc` : analyse un commit distant (nécessite token GitHub)
> - Mode `-gp` : analyse une PR entière (tous les commits)
> - Commencez par tester avec un petit échantillon (`SAMPLE_SIZE = 5`)
> - Sauvegardez les résultats intermédiaires en JSON pour éviter de tout relancer

In [ ]:
# Étape 2.4 — Test de RefactoringMiner
# RMINER_CMD est défini dans la cellule 0.2

OUTPUT_RM_DIR = os.path.join("outputs", "rminer")
os.makedirs(OUTPUT_RM_DIR, exist_ok=True)

# Test avec le dépôt public officiel de l'auteur de RefactoringMiner
TEST_URL = "https://github.com/danilofes/refactoring-toy-example.git"
TEST_SHA = "36287f7c3b09eff78395267a3ac0d7da067863fd"
TEST_OUT = os.path.join(OUTPUT_RM_DIR, "test_output.json")

cmd = RMINER_CMD + ["-gc", TEST_URL, TEST_SHA, "60", "-json", TEST_OUT]
print("Commande :", " ".join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
print("Return code:", result.returncode)

if os.path.exists(TEST_OUT):
    with open(TEST_OUT) as f:
        test_data = json.load(f)
    n = sum(len(c.get("refactorings", [])) for c in test_data.get("commits", []))
    print(f"✅ Test réussi — {n} refactoring(s) détecté(s) dans le commit de test")
else:
    print("❌ Fichier JSON non créé")
    print("stdout:", result.stdout[:400])
    print("stderr:", result.stderr[:400])

In [ ]:
# Étape 2.4 — Fonctions run_refactoringminer_commit() et run_refactoringminer_pr()

def run_refactoringminer_commit(repo_full_name, sha, timeout=120):
    """Analyse un commit avec RefactoringMiner -gc. Retourne le dict JSON ou None."""
    out_file = os.path.join(OUTPUT_RM_DIR, f"commit_{sha[:12]}.json")
    if os.path.exists(out_file):
        with open(out_file) as f:
            return json.load(f)
    repo_url = f"https://github.com/{repo_full_name}.git"
    cmd = RMINER_CMD + ["-gc", repo_url, sha, str(timeout), "-json", out_file]
    try:
        subprocess.run(cmd, capture_output=True, text=True, timeout=timeout + 10)
        if os.path.exists(out_file):
            with open(out_file) as f:
                return json.load(f)
        return None
    except subprocess.TimeoutExpired:
        print(f"  ⏱ Timeout: {repo_full_name} @ {sha[:8]}")
        return None
    except Exception as e:
        print(f"  ❌ Erreur: {e}")
        return None


def run_refactoringminer_pr(repo_full_name, pr_number, timeout=300):
    """Analyse une PR entière avec RefactoringMiner -gp. Retourne le dict JSON ou None."""
    safe_name = repo_full_name.replace("/", "_")
    out_file = os.path.join(OUTPUT_RM_DIR, f"pr_{safe_name}_{pr_number}.json")
    if os.path.exists(out_file):
        with open(out_file) as f:
            return json.load(f)
    repo_url = f"https://github.com/{repo_full_name}.git"
    cmd = RMINER_CMD + ["-gp", repo_url, str(pr_number), str(timeout), "-json", out_file]
    try:
        subprocess.run(cmd, capture_output=True, text=True, timeout=timeout + 30)
        if os.path.exists(out_file):
            with open(out_file) as f:
                return json.load(f)
        return None
    except subprocess.TimeoutExpired:
        print(f"  ⏱ Timeout: {repo_full_name} PR#{pr_number}")
        return None
    except Exception as e:
        print(f"  ❌ Erreur: {e}")
        return None


print("✅ Fonctions run_refactoringminer_commit() et run_refactoringminer_pr() définies")

In [ ]:
# Étape 2.4 — Boucle d'analyse RefactoringMiner (parallèle)
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

SAMPLE_SIZE = 684   # augmentez après validation (ex: 50, puis 684)
MAX_WORKERS = 8   # workers parallèles (GitHub API)
CHECKPOINT_FILE = os.path.join(OUTPUT_RM_DIR, "rminer_results.json")

# Charger checkpoint si disponible
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        rminer_results = json.load(f)
    print(f"[checkpoint] {len(rminer_results)} PRs déjà analysées")
else:
    rminer_results = {}

pr_number_col = 'number'
repo_name_col = 'full_name'

sample_prs = df_prs_refactoring.head(SAMPLE_SIZE)
print(f"\nAnalyse de {SAMPLE_SIZE} PRs avec RefactoringMiner (-gp) — {MAX_WORKERS} workers...")

lock = threading.Lock()

def analyse_pr(pr_row):
    pr_id = str(pr_row['id_pr'])
    with lock:
        if pr_id in rminer_results:
            return pr_id, None
    pr_num = int(pr_row[pr_number_col])
    repo   = str(pr_row[repo_name_col])
    print(f"  → {repo} PR#{pr_num}")
    result = run_refactoringminer_pr(repo, pr_num, timeout=300)
    return pr_id, result

rows = [row for _, row in sample_prs.iterrows()]

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(analyse_pr, row): row for row in rows}
    for i, future in enumerate(tqdm(as_completed(futures), total=len(rows), desc="PRs"), 1):
        pr_id, result = future.result()
        if result is not None:
            with lock:
                rminer_results[pr_id] = result
        if i % 5 == 0:
            with lock:
                with open(CHECKPOINT_FILE, 'w') as f:
                    json.dump(rminer_results, f)

with open(CHECKPOINT_FILE, 'w') as f:
    json.dump(rminer_results, f)

analyzed = sum(1 for v in rminer_results.values() if v is not None)
print(f"\n✅ {analyzed}/{len(sample_prs)} PRs analysées")
for pr_id, res in list(rminer_results.items())[:3]:
    if res:
        n_ref = sum(len(c.get('refactorings', [])) for c in res.get('commits', []))
        print(f"  PR {pr_id}: {n_ref} refactoring(s) détecté(s)")

### Étape 2.5 — Croiser les résultats et classifier

Pour chaque commentaire de refactoring, vérifiez si un refactoring a été détecté dans un commit de suivi touchant le même fichier.

> **Indice — Schéma de classification :**
> ```
> Pour chaque commentaire de refactoring :
>   1. Y a-t-il des commits de suivi ?
>      NON → "PR abandonnée" (si non fusionnée) ou "Non appliqué"
>   2. RefactoringMiner détecte-t-il un refactoring dans le même fichier ?
>      OUI → "Appliqué (confirmé)"
>      NON → "Appliqué (non confirmé)" ou "Non appliqué"
> ```
>
> Dans la sortie RefactoringMiner, les fichiers sont dans `leftSideLocations[0].filePath` et `rightSideLocations[0].filePath`.

In [ ]:
# TODO : Écrire classify_application(comment_path, followup_commits, rminer_results, pr_merged)
#
# INDICES :
#   1. Si pas de followup_commits → 'PR abandonnée' ou 'Non appliqué'
#   2. Pour chaque SHA de suivi, cherchez les refactorings dans rminer_results
#   3. Comparez comment_path avec les filePath dans les refactorings détectés
#   4. Retournez la catégorie appropriée

# VOTRE CODE ICI

In [ ]:
# TODO : Appliquer la classification sur tous les commentaires de refactoring
#
# INDICES :
#   1. Construisez un index {sha: [refactorings]} depuis rminer_results
#   2. Pour chaque PR analysée, appelez get_followup_commits()
#   3. Pour chaque commentaire, appelez classify_application()
#   4. Stockez dans une liste de dicts → DataFrame df_results
#   5. Affichez df_results['classification'].value_counts()

# VOTRE CODE ICI

### Étape 2.6 — Calculer les taux d'application

> **Indice :** Utilisez `pd.crosstab()` pour un tableau agent × classification. Normalisez par ligne pour obtenir des pourcentages. Visualisez avec un barplot empilé.

In [ ]:
# TODO : Diagramme en barres empilées par agent + comparaison Java vs Python
#
# INDICES :
#   ct = pd.crosstab(df_results['agent'], df_results['classification'])
#   ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
#   ct_pct.plot(kind='bar', stacked=True, colormap='Set2')

# VOTRE CODE ICI

### Étape 2.7 (Bonus) — Correspondance sémantique

Pour les cas « appliqué confirmé », vérifiez si le type de refactoring détecté correspond à la suggestion.

> **Indice :** Créez un mapping entre les mots-clés de la suggestion et les types RefactoringMiner. Par exemple, le mot `'extract'` dans un commentaire devrait correspondre à `'Extract Method'`, `'Extract Class'`, etc. dans les résultats.

In [ ]:
# TODO (Bonus) : Vérifier la correspondance sémantique
#
# INDICE :
#   SUGGESTION_TO_RMINER = {
#       'extract': ['Extract Method', 'Extract Class', ...],
#       'rename':  ['Rename Method', 'Rename Variable', ...],
#       'inline':  ['Inline Method', 'Inline Variable'],
#       ...
#   }
#
#   def check_semantic_match(comment_body, detected_type):
#       # Vérifiez si un mot-clé du commentaire correspond au type détecté
#       ...

# VOTRE CODE ICI

---
## Annexe : Fonctions utilitaires

Quelques fonctions utiles que vous pouvez réutiliser.

In [ ]:
def safe_json_load(filepath):
    """Charge un fichier JSON en gérant les erreurs."""
    try:
        with open(filepath) as f:
            return json.load(f)
    except (json.JSONDecodeError, FileNotFoundError) as e:
        print(f"Erreur chargement {filepath}: {e}")
        return None


def extract_refactorings_from_rminer(result_json):
    """
    Extrait la liste plate de refactorings depuis la sortie RefactoringMiner.

    Returns:
        list of dict: [{sha, type, description, filePath}, ...]
    """
    refactorings = []
    if not result_json:
        return refactorings
    for commit in result_json.get('commits', []):
        sha = commit.get('sha1', '')
        for ref in commit.get('refactorings', []):
            left_files = [loc.get('filePath', '') for loc in ref.get('leftSideLocations', [])]
            right_files = [loc.get('filePath', '') for loc in ref.get('rightSideLocations', [])]
            refactorings.append({
                'sha': sha,
                'type': ref.get('type', ''),
                'description': ref.get('description', ''),
                'leftFiles': left_files,
                'rightFiles': right_files,
            })
    return refactorings

print("Fonctions utilitaires chargées.")